# 202

#### 5. Single plots

In [ ]:
# NOTE: This code already is in the 203 notebook

# Sort once so all hydrologic plots draw in a consistent left-to-right order.
# The x-axis is effective total pumping because the fish-dollars forward model
# can shut off farms when pumping drops below the 70% rule.
plot_df = up.sort_for_plot(member_summary, PUMPING_COLUMN_FOR_HYDRO_PLOTS)

In [ ]:
# =============================================================================
# Figure 3: Expected streamflow and uncertainty band
# =============================================================================
# This is the most important hydrologic plot in Notebook 202.
#
# It compares:
#   1. the baseline-T streamflow prediction,
#   2. the probability-weighted expected streamflow across uncertain T values, and
#   3. the weighted 5th to 95th percentile streamflow range.
#
# The shaded band is useful because it tells us how much the streamflow result
# could vary because T is uncertain, even though the pumping design is fixed.
fig, ax = plt.subplots()
ax.plot(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["baseline_T_streamflow_cfs"],
    linestyle="--",
    label="Baseline-T streamflow",
    color=up.SCENARIO_COLORS["baseline_T"],
)
ax.plot(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["expected_streamflow_cfs"],
    label="Probability-weighted expected streamflow",
    color=up.SCENARIO_COLORS["expected"],
)
ax.fill_between(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["streamflow_p05_cfs"],
    plot_df["streamflow_p95_cfs"],
    alpha=0.22,
    color=up.SCENARIO_COLORS["uncertainty_band"],
    label="Weighted 5th–95th percentile range",
)
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Streamflow = 8.6 cfs - depletion (cfs)")
ax.set_title("Unknown T: probability-weighted streamflow uncertainty")
ax.legend()
up.save_figure(fig, OUTPUT_DIR / "202_expected_streamflow_with_uncertainty_band.png")

# =============================================================================
# Figure 4: Probability-weighted absolute streamflow error
# =============================================================================
# This metric answers:
#   "On average, how large is the streamflow prediction error caused by uncertain T?"
#
# It treats over-prediction and under-prediction as equally important because it
# uses the absolute value of the streamflow error.
fig, ax = plt.subplots()
ax.plot(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["probability_weighted_absolute_streamflow_error_cfs"],
    marker="o",
    markersize=3,
    color=up.SCENARIO_COLORS["expected"],
)
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted absolute streamflow error (cfs)")
ax.set_title("Expected magnitude of streamflow error from T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_probability_weighted_absolute_streamflow_error.png")

# =============================================================================
# Figure 5: Probability-weighted streamflow shortfall
# =============================================================================
# This is the most management-focused cost metric.
#
# It only counts cases where uncertain T predicts LESS streamflow than the
# baseline-T prediction. In other words, it asks:
#   "What is the expected streamflow loss if the assumed T is wrong?"
fig, ax = plt.subplots()
ax.plot(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["probability_weighted_streamflow_shortfall_cfs"],
    marker="o",
    markersize=3,
    color=up.SCENARIO_COLORS["T_plus_10pct"],
)
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Probability-weighted streamflow shortfall (cfs)")
ax.set_title("Expected one-sided streamflow loss from T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_probability_weighted_streamflow_shortfall.png")

# =============================================================================
# Figure 6: Streamflow standard deviation
# =============================================================================
# This plot is a compact way to show spread in streamflow predictions caused by
# T uncertainty. It does not distinguish between beneficial and harmful changes;
# it is simply the weighted standard deviation of streamflow across T values.
fig, ax = plt.subplots()
ax.plot(
    plot_df[PUMPING_COLUMN_FOR_HYDRO_PLOTS],
    plot_df["streamflow_std_cfs"],
    marker="o",
    markersize=3,
    color="0.35",
)
ax.set_xlabel("Effective total pumping after fish-dollars cutoff (cfs)")
ax.set_ylabel("Streamflow standard deviation (cfs)")
ax.set_title("Spread in streamflow caused by T uncertainty")
up.save_figure(fig, OUTPUT_DIR / "202_streamflow_standard_deviation.png")

# 203

## 5. Fixed-design known T-error figures

These plots use the **same baseline Pareto designs** and re-evaluate them under different T assumptions. Total pumping does not change between scenarios because the pumping designs are held fixed.

These figures answer: **If the selected baseline designs are kept, how much would streamflow/depletion change if T were actually different?**


In [ ]:
# -----------------------------------------------------------------------------
# Known T-error plots from Notebook 201 outputs.
# -----------------------------------------------------------------------------
# Streamflow view: this is usually the most intuitive for management discussion.
up.plot_scenario_lines(
    known_wide,
    xcol=PUMPING_COLUMN_FOR_HYDRO_PLOTS,
    ycols_by_scenario={
        "T_minus_10pct": "streamflow_cfs__T_minus_10pct",
        "baseline_T": "streamflow_cfs__baseline_T",
        "T_plus_10pct": "streamflow_cfs__T_plus_10pct",
    },
    xlabel="Effective total pumping after fish-dollars cutoff (cfs)",
    ylabel="Streamflow = 8.6 cfs - depletion (cfs)",
    title="Fixed baseline designs re-evaluated under known T error",
    outfile=OUTPUT_DIR / "203_known_T_error_pumping_vs_streamflow.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)

# Depletion view: this is the same information as streamflow, but shown as the
# hydrologic impact subtracted from the historical/reference streamflow.
up.plot_scenario_lines(
    known_wide,
    xcol=PUMPING_COLUMN_FOR_HYDRO_PLOTS,
    ycols_by_scenario={
        "T_minus_10pct": "depletion_cfs__T_minus_10pct",
        "baseline_T": "depletion_cfs__baseline_T",
        "T_plus_10pct": "depletion_cfs__T_plus_10pct",
    },
    xlabel="Effective total pumping after fish-dollars cutoff (cfs)",
    ylabel="Streamflow depletion (cfs)",
    title="Fixed baseline designs: depletion response to known T error",
    outfile=OUTPUT_DIR / "203_known_T_error_pumping_vs_depletion.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)


## 7. Original 05_MOU re-optimized front comparison

This section loads your original tradeoff CSVs for the separate `05_MOU` runs:

- baseline T,
- T −10%, and
- T +10%.

These are the **fully re-optimized fronts**. They are different from the fixed-design results above because PEST++ MOU was allowed to find a new front for each T case.


In [ ]:
# -----------------------------------------------------------------------------
# Load original fully optimized MOU tradeoff fronts.
# -----------------------------------------------------------------------------
original_tradeoffs = up.load_original_mou_tradeoff_fronts(
    project_dir=PROJECT_DIR,
    notebook_dir=NOTEBOOK_DIR,
    tradeoff_files=ORIGINAL_MOU_TRADEOFF_FILES,
)

print("Original MOU tradeoff rows loaded:", len(original_tradeoffs))

if not original_tradeoffs.empty:
    original_summary = (
        original_tradeoffs.groupby("scenario", sort=False)
        .agg(
            n_points=("total_pumping_cfs", "count"),
            min_pumping_cfs=("total_pumping_cfs", "min"),
            max_pumping_cfs=("total_pumping_cfs", "max"),
            mean_streamflow_cfs=("streamflow_cfs", "mean"),
            mean_depletion_cfs=("depletion_cfs", "mean"),
        )
        .reset_index()
    )
    original_summary.to_csv(OUTPUT_DIR / "203_original_MOU_tradeoff_front_summary.csv", index=False)
    original_tradeoffs.to_csv(OUTPUT_DIR / "203_original_MOU_tradeoff_fronts_loaded.csv", index=False)
    display(original_summary)
else:
    print("No original MOU tradeoff CSVs were found. The original-front plots will be skipped.")

# Original optimized fronts: streamflow view.
up.plot_original_mou_tradeoffs(
    original_tradeoffs,
    ycol="streamflow_cfs",
    ylabel="Streamflow = 8.6 cfs - depletion (cfs)",
    title="Original 05_MOU optimized fronts: streamflow vs pumping",
    outfile=OUTPUT_DIR / "203_original_MOU_fronts_pumping_vs_streamflow.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)

# Original optimized fronts: depletion view.
up.plot_original_mou_tradeoffs(
    original_tradeoffs,
    ycol="depletion_cfs",
    ylabel="Streamflow depletion (cfs)",
    title="Original 05_MOU optimized fronts: depletion vs pumping",
    outfile=OUTPUT_DIR / "203_original_MOU_fronts_pumping_vs_depletion.png",
    colors=SCENARIO_COLORS,
    labels=SCENARIO_LABELS,
)
